In [ ]:
import os
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

FEATURE_SIZES = [40000, 5000, 1000]
os.makedirs("outputs", exist_ok=True)

In [ ]:
TRAIN_PATH = "train_split.csv"
TEST_PATH = "test_split.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
train_df["clean_text"] = train_df["clean_text"].fillna("")
test_df["clean_text"] = test_df["clean_text"].fillna("")

print(f"Loaded train split: {train_df.shape[0]} rows")
print(f"Loaded test split : {test_df.shape[0]} rows")

Loaded train split: 6400 rows
Loaded test split : 1600 rows


In [ ]:
def build_bow_features(train_df, test_df, max_features):
    vectorizer = CountVectorizer(max_features=max_features)
    X_train = vectorizer.fit_transform(train_df["clean_text"])
    X_test = vectorizer.transform(test_df["clean_text"])
    return vectorizer, X_train, X_test

def sparsity(matrix):
    nonzero = matrix.nnz
    total = matrix.shape[0] * matrix.shape[1]
    return 100 * (1 - nonzero / total)

In [ ]:
results = []

for max_feat in FEATURE_SIZES:
    vectorizer, X_train, X_test = build_bow_features(train_df, test_df, max_feat)
    vocab_size = len(vectorizer.vocabulary_)
    train_sparsity = sparsity(X_train)

    print(f"\n--- max_features = {max_feat} ---")
    print(f"Vocabulary size (actual)   : {vocab_size}")
    print(f"Train matrix shape         : {X_train.shape}")
    print(f"Test matrix shape          : {X_test.shape}")
    print(f"Train matrix sparsity      : {train_sparsity:.2f}% zeros")
    print(f"Non-zero entries (train)   : {X_train.nnz}")
    print("Sample vocabulary (first 15 terms, alphabetical):")
    print(sorted(vectorizer.vocabulary_.keys())[:15])

    results.append({
        "max_features_setting": max_feat,
        "actual_vocab_size": vocab_size,
        "train_matrix_shape": str(X_train.shape),
        "test_matrix_shape": str(X_test.shape),
        "train_sparsity_pct": round(train_sparsity, 2),
        "train_nonzero_entries": X_train.nnz,
    })

results_df = pd.DataFrame(results)
print("\nSUMMARY TABLE")
print(results_df.to_string(index=False))


--- max_features = 40000 ---
Vocabulary size (actual)   : 40000
Train matrix shape         : (6400, 40000)
Test matrix shape          : (1600, 40000)
Train matrix sparsity      : 99.77% zeros
Non-zero entries (train)   : 600241
Sample vocabulary (first 15 terms, alphabetical):
['aa', 'aaa', 'aalrl', 'aame', 'aap', 'aaron', 'aarons', 'aarp', 'ab', 'aba', 'aback', 'abacus', 'abandon', 'abandoned', 'abandonedbag']

--- max_features = 5000 ---
Vocabulary size (actual)   : 5000
Train matrix shape         : (6400, 5000)
Test matrix shape          : (1600, 5000)
Train matrix sparsity      : 98.43% zeros
Non-zero entries (train)   : 502470
Sample vocabulary (first 15 terms, alphabetical):
['abandon', 'abandoned', 'ability', 'able', 'abroad', 'abruptly', 'absence', 'absent', 'absolute', 'absolutely', 'absorb', 'abuse', 'academic', 'accelerate', 'accelerated']

--- max_features = 1000 ---
Vocabulary size (actual)   : 1000
Train matrix shape         : (6400, 1000)
Test matrix shape          : (1

In [ ]:
full_vocab = results_df.iloc[0]["actual_vocab_size"]
print(f"The true vocabulary of the cleaned training text has at least {full_vocab} unique tokens.")

for _, row in results_df.iterrows():
    capped = row["actual_vocab_size"] < row["max_features_setting"]
    note = ("(the requested cap was never reached)" if capped
            else "(vocabulary was capped here, keeping only the most frequent terms)")
    print(f"  max_features={row['max_features_setting']:>6} -> "
          f"{row['actual_vocab_size']} columns, matrix {row['train_matrix_shape']}, "
          f"{row['train_sparsity_pct']}% sparse {note}")

print("\nAs max_features decreases from 40,000 to 5,000 to 1,000, CountVectorizer "
      "keeps only the most frequent remaining terms and drops rarer words. This "
      "shrinks the number of matrix columns (less memory, faster training) but "
      "risks discarding domain-specific vocabulary that could matter for "
      "distinguishing Relevant articles.")

The true vocabulary of the cleaned training text has at least 40000 unique tokens.
  max_features= 40000 -> 40000 columns, matrix (6400, 40000), 99.77% sparse (vocabulary was capped here, keeping only the most frequent terms)
  max_features=  5000 -> 5000 columns, matrix (6400, 5000), 98.43% sparse (vocabulary was capped here, keeping only the most frequent terms)
  max_features=  1000 -> 1000 columns, matrix (6400, 1000), 94.8% sparse (vocabulary was capped here, keeping only the most frequent terms)

As max_features decreases from 40,000 to 5,000 to 1,000, CountVectorizer keeps only the most frequent remaining terms and drops rarer words. This shrinks the number of matrix columns (less memory, faster training) but risks discarding domain-specific vocabulary that could matter for distinguishing Relevant articles.


In [ ]:
from google.colab import files

out_path = "outputs/feature_extraction_summary.csv"
results_df.to_csv(out_path, index=False)
files.download(out_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>